## Jak działa partycjonowanie

1. Rozpocznij z 8 partycjami.
2. Uruchom kod.
3. Otwórz **Spark UI**
4. Sprawdź drugi job (czy są jakieś różnice pomięczy drugim)
5. Sprawdź **Event Timeline**
6. Sprawdzaj czas wykonania.
  * Uruchom kilka razy rzeby sprawdzić średni czas wykonania.

Powtórz z inną liczbą partycji
* 1 partycja
* 7 partycja
* 9 partycja
* 16 partycja
* 24 partycja
* 96 partycja
* 200 partycja
* 4000 partycja

Zastąp `repartition(n)` z `coalesce(n)` używając:
* 6 partycji
* 5 partycji
* 4 partycji
* 3 partycji
* 2 partycji
* 1 partycji

***Note:*** *Dane muszą być wystarczająco duże żeby zaobserwować duże różnice z małymi partycjami.*<br/>* To co możesz sprawdzić jak zachowują się małe dane z dużą ilośćia partycji.*

In [0]:
# slots = sc.defaultParallelism
spark.conf.get("spark.sql.shuffle.partitions")

Out[1]: '200'

In [0]:
spark.catalog.clearCache()
parquetDir = "/FileStore/tables/training/wikipedia/pageviews/"

df = (spark.read
  .parquet(parquetDir)
  .repartition(2000)
#    .coalesce(6)
  .groupBy("count_views").sum())


df.explain
df.count()

Out[5]: 966

In [0]:
df.rdd.getNumPartitions()

Out[8]: 5

In [0]:
import time
# Ścieżka do danych Parquet
parquetDir = "/FileStore/tables/training/wikipedia/pageviews/"
df = spark.read.parquet(parquetDir)

# Sprawdzenie początkowej liczby partycji
print("Początkowa liczba partycji:", df.rdd.getNumPartitions())

# REPARTITION – powoduje shuffle
df_repartitioned = df.repartition(9) 
print("Po repartition:", df_repartitioned.rdd.getNumPartitions())

start_time = time.time()
df_repartitioned.groupBy("count_views").sum().count()
print("Czas wykonania z repartition:", round(time.time() - start_time, 2), "s")

# COALESCE – mniej partycji, bez shuffle
df_coalesced = df.coalesce(1)
print("Po coalesce:", df_coalesced.rdd.getNumPartitions())

start_time = time.time()
df_coalesced.groupBy("count_views").sum().count()
print("Czas wykonania z coalesce:", round(time.time() - start_time, 2), "s")

Początkowa liczba partycji: 8
Po repartition: 9
Czas wykonania z repartition: 7.5 s
Po coalesce: 1
Czas wykonania z coalesce: 2.66 s


**repartition** 

- 4000 - 334.75s 316.56 - **325s**
- 200 - 16.62 15.84 15.55 - **16s**
- 96 - 10.53 10.58 - **10.55s**
- 24 - 7.26 7.3 8.04- **7.53s**
- 16 - 6.58 7.09 6.56 - **6.74s**
- 9 - 6.49 6.22 7.5 - **6.73s**
- 7 - 6.01 5.74 6.37 - **6.04s**
- 1 - 3.98s 3.69 3.9 - **3.56s**

**coalesce**

- 1 - 2.78 2.53 2.98    -    **2.77s**
- 2 - 2.90 2.08 2.04    -    **2.34s**
- 3 - 2.20 2.27 2.50    -    **2.32s**
- 4 - 3.08 2.50 2.25    -    **2.61s**
- 5 - 2.24 2.32 2.0 2.03 -   **2.15s**
- 6 - 2.46 2.41 2.97    -    **2.8s**